In [1]:
import faiss
import orjson

from llama_index.core import (
    Settings,
    SimpleDirectoryReader,
    load_index_from_storage,
    VectorStoreIndex,
    StorageContext,
    get_response_synthesizer,
)
from llama_index.core.llms import ChatMessage
from llama_index.core.schema import (
    Document,
    TextNode,
)
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.readers.json import JSONReader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from IPython.display import Markdown, display

/home/ed/miniconda3/envs/mypy311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Setup

In [2]:
text_fields = ["content"]
metadata_fields = ["title", "date"]


d = 2304 # for gemma-2-embed
path = "/mnt/d/temp/user/ed/mart/mmplastic/20240725_154000_000000"

index_path = "/mnt/d/temp/user/ed/warehouse/mmplastic/vector_index.faiss/20240726_152000_000000"

faiss_index = faiss.IndexFlatL2(d)
faiss_index.verbose=True


In [3]:
llm = Ollama(
    model='tiger-gemma2',
    request_timeout=30000.0,
    keep_alive="10m",
    additional_kwargs={"mirostat": 0, "keep_alive": "10m"}
)

Settings.llm = llm


In [4]:
embeddings = OllamaEmbedding(
    model_name="gemma-2-embed",
    base_url="http://localhost:11434",
    additional_kwargs={"mirostat": 0, "keep_alive": "10m", "num_gpu": -1},
    request_timeout=3000.0,
    keep_alive="10m",
)
# embeddings = HuggingFaceEmbedding(
#     model_name="klue/bert-base"
# )
Settings.embed_model = embeddings

In [5]:
llm.__dict__

{'callback_manager': <llama_index.core.callbacks.base.CallbackManager at 0x7efc1bc2ff10>,
 'system_prompt': None,
 'messages_to_prompt': <function llama_index.core.base.llms.generic_utils.messages_to_prompt(messages: Sequence[llama_index.core.base.llms.types.ChatMessage]) -> str>,
 'completion_to_prompt': <function llama_index.core.llms.llm.default_completion_to_prompt(prompt: str) -> str>,
 'output_parser': None,
 'pydantic_program_mode': <PydanticProgramMode.DEFAULT: 'default'>,
 'query_wrapper_prompt': None,
 'base_url': 'http://localhost:11434',
 'model': 'tiger-gemma2',
 'temperature': 0.75,
 'context_window': 3900,
 'request_timeout': 30000.0,
 'prompt_key': 'prompt',
 'json_mode': False,
 'additional_kwargs': {'mirostat': 0, 'keep_alive': '10m'},
 'is_function_calling_model': True}

In [6]:
documents = SimpleDirectoryReader(path).load_data()
# documents = JSONReader(
#     ensure_ascii=False,
#     is_jsonl=None
# ).load_data(path + "/data.json", extra_info={})

In [37]:
documents[0].__dict__

{'id_': '9028481c-8c5f-418d-9c0a-0e162ce35538',
 'embedding': None,
 'metadata': {'file_path': '/mnt/d/temp/user/ed/mart/mmplastic/20240725_154000_000000/data.json',
  'file_name': 'data.json',
  'file_type': 'application/json',
  'file_size': 10529,
  'creation_date': '2024-08-02',
  'last_modified_date': '2024-08-02'},
 'excluded_embed_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'excluded_llm_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'relationships': {},
 'text': '[\n{\n"title": "미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착",\n"content": "미세플라스틱(Microplastic)이 환경에서 노후화가 진행됨에 따라 더 많은 유해화학물질을 흡수하는 것으로 밝혀졌다. 이는 미국 애리조나 주립대학이 수행한 연구에 따라 확인되었으며, 연구 결과에 따르면 미세플라스틱은 햇빛 등에 노출 시 표면입자가 변형되어 유기오염물질을 더 잘 흡수할 수 있는 상태로 변형되는 것으로 밝혀졌다.\\n\\n연구팀은 폴리에틸렌(Polyethylene) 및 폴리프로필렌(Polypropylene)을 사용하여 미세플라스틱을 인위적으로 생성하고, 자외선 및 과산화수소를 이용해 풍화시킨 후, 풍화된 미세플

In [42]:
question = "미세플라스틱이 햇빛에 노출되면 오염물질을 흡수하는 연구와 관련된 미국 대학교를 찾고있어"

messages = [
    ChatMessage(
        role="system", content="You are a helpful assistant. 한글로만 답변해"
    ),
    ChatMessage(
        role="user", content=question
    ),
]

In [45]:
keyword_response = llm.complete(f"아래 질문에서 누가, 무엇을, 어떻게, 이유 등과 관련된 가장 중요한 키워드만 숫자없이 답변해줘\n{question}")

In [46]:
keyword_response.text

'미세 플라스틱, 햇빛, 오염물질, 미국 대학교'

In [47]:
documents

[Document(id_='9028481c-8c5f-418d-9c0a-0e162ce35538', embedding=None, metadata={'file_path': '/mnt/d/temp/user/ed/mart/mmplastic/20240725_154000_000000/data.json', 'file_name': 'data.json', 'file_type': 'application/json', 'file_size': 10529, 'creation_date': '2024-08-02', 'last_modified_date': '2024-08-02'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, text='[\n{\n"title": "미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착",\n"content": "미세플라스틱(Microplastic)이 환경에서 노후화가 진행됨에 따라 더 많은 유해화학물질을 흡수하는 것으로 밝혀졌다. 이는 미국 애리조나 주립대학이 수행한 연구에 따라 확인되었으며, 연구 결과에 따르면 미세플라스틱은 햇빛 등에 노출 시 표면입자가 변형되어 유기오염물질을 더 잘 흡수할 수 있는 상태로 변형되는 것으로 밝혀졌다.\\n\\n연구팀은 폴리에틸렌(Polyethylene) 및 폴리프로필렌(Polypropylene)을 사용하여 미세플라스틱을 인위적으로 생성하고, 자외선 및 과산화수소를 이용해 풍화시킨 후, 풍화된 미세플라스틱 입자를 오염물질 모델인 페난트렌(phenanthrene)및 메틸렌 블루(meth

In [11]:
len(documents)

1

In [12]:
type(documents[0])

llama_index.core.schema.Document

In [48]:
doc = documents[0]

doc.text

'[\n{\n"title": "미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착",\n"content": "미세플라스틱(Microplastic)이 환경에서 노후화가 진행됨에 따라 더 많은 유해화학물질을 흡수하는 것으로 밝혀졌다. 이는 미국 애리조나 주립대학이 수행한 연구에 따라 확인되었으며, 연구 결과에 따르면 미세플라스틱은 햇빛 등에 노출 시 표면입자가 변형되어 유기오염물질을 더 잘 흡수할 수 있는 상태로 변형되는 것으로 밝혀졌다.\\n\\n연구팀은 폴리에틸렌(Polyethylene) 및 폴리프로필렌(Polypropylene)을 사용하여 미세플라스틱을 인위적으로 생성하고, 자외선 및 과산화수소를 이용해 풍화시킨 후, 풍화된 미세플라스틱 입자를 오염물질 모델인 페난트렌(phenanthrene)및 메틸렌 블루(methylene blue)에 노출시켰다. 표면의 풍화는 표면적 및 표면화학 성질을 변화시켜 오염물질의 흡착을 증가시키는 결과를 나타냈다.\\n\\n연구팀의 모델 시스템은 미세플라스틱 노후화의 초기 단계를 보여주지만, 실제 미세플라스틱은 환경에서 수 십 년 동안 풍화되므로, 실험조건보다 더욱 노화된 미세플라스틱이 실제 환경에서 유기오염물질과의 상호작용을 통해 더욱 악화된 환경위해를 초래할 수 있는지에 대한 시사점을 제시하고 있다.\\n\\n연구팀은Chemospere저널에 발표한 논문에서, ‘환경 내에서 오염물질의 매개체로써의 미세플라스틱의 잠재적인 역할을 고려할 때, 미세플라스틱이 야기하는 미래의 환경 위험성을 평가하기 위해서는 표면 풍화와 오염물질 흡착에 대한 복잡한 상호작용을 이해하는 것이 매우 중요하다.‘‘고 결론지었다.\\n\\nPerreault 박사는 ‘이번 연구는 환경 내에서의 풍화작용이 미세플라스틱의 거동에 어떤 영향을 미치는 지에 관한 장기 프로젝트의 첫 단계이며, 보다 현실적인 미세플라스틱 모델을 구현함으로써, 햇빛이나 미생물로 인한 환경변화가 미세플라스틱과 오염물질 간 상호작용에 미치는 영향을 살펴볼 것이다.‘라고 밝혔다. 앞으로 연구팀

In [14]:
text_splitter = RecursiveCharacterTextSplitter(
    separators="\n\n",
    chunk_size=200,
    chunk_overlap=0,
    keep_separator=True
)

nodes = [
    TextNode(metadata=dict([(m, doc[m]) for m in metadata_fields] + [('seq_num', seq_num), ('sub_seq_num', sub_seq_num)]), text=text)
    for seq_num, doc in enumerate(orjson.loads(doc.text)) for sub_seq_num, text in enumerate(text_splitter.split_text("\n".join([doc[t] for t in text_fields])))
]

In [15]:
nodes

[TextNode(id_='32d7cb50-94ff-46ae-a1be-9aedfc6435d1', embedding=None, metadata={'title': '미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착', 'date': '2022-04-09', 'seq_num': 0, 'sub_seq_num': 0}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, text='미세플라스틱(Microplastic)이 환경에서 노후화가 진행됨에 따라 더 많은 유해화학물질을 흡수하는 것으로 밝혀졌다. 이는 미국 애리조나 주립대학이 수행한 연구에 따라 확인되었으며, 연구 결과에 따르면 미세플라스틱은 햇빛 등에 노출 시 표면입자가 변형되어 유기오염물질을 더 잘 흡수할 수 있는 상태로 변형되는 것으로 밝혀졌다.', mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}', metadata_template='{key}: {value}', metadata_seperator='\n'),
 TextNode(id_='99b3652a-d646-4b11-8236-8421fcb439c4', embedding=None, metadata={'title': '미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착', 'date': '2022-04-09', 'seq_num': 0, 'sub_seq_num': 1}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, text='\n연구팀은 폴리에틸렌(Polyethylene) 및 폴리프로필렌(Polypropylene)을 사용하여 미세플라스틱을 인위적으로 생성하고, 자외선 및 과산화수소를 이용해 풍화시킨 후, 풍화된 미세플라스틱 입자

In [16]:
docstore = SimpleDocumentStore()
docstore.add_documents(nodes)

In [17]:
faiss_index

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x7efd3ba12160> >

In [ ]:
faiss_index.reset()

vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store, docstore=docstore)

# Querying

In [19]:
index = VectorStoreIndex(
    nodes=nodes, storage_context=storage_context
)
# index.storage_context.persist(index_path)


#### ollama_request_body: {'prompt': 'title: 미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착\ndate: 2022-04-09\nseq_num: 0\nsub_seq_num: 0\n\n미세플라스틱(Microplastic)이 환경에서 노후화가 진행됨에 따라 더 많은 유해화학물질을 흡수하는 것으로 밝혀졌다. 이는 미국 애리조나 주립대학이 수행한 연구에 따라 확인되었으며, 연구 결과에 따르면 미세플라스틱은 햇빛 등에 노출 시 표면입자가 변형되어 유기오염물질을 더 잘 흡수할 수 있는 상태로 변형되는 것으로 밝혀졌다.', 'model': 'gemma-2-embed', 'options': {}}
#### ollama_request_body: {'prompt': 'title: 미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착\ndate: 2022-04-09\nseq_num: 0\nsub_seq_num: 1\n\n\n연구팀은 폴리에틸렌(Polyethylene) 및 폴리프로필렌(Polypropylene)을 사용하여 미세플라스틱을 인위적으로 생성하고, 자외선 및 과산화수소를 이용해 풍화시킨 후, 풍화된 미세플라스틱 입자를 오염물질 모델인 페난트렌(phenanthrene)및 메틸렌 블루(methylene blue)에 노출시켰다. 표면의 풍화는 표면적 및 표면화학 성질을 변화시켜 오염물질의 흡착을 증가시키는 결과를 나타냈다.', 'model': 'gemma-2-embed', 'options': {}}
#### ollama_request_body: {'prompt': 'title: 미세플라스틱, 노후화 될수록 더 많은 오염물질 흡착\ndate: 2022-04-09\nseq_num: 0\nsub_seq_num: 2\n\n연구팀의 모델 시스템은 미세플라스틱 노후화의 초기 단계를 보여주지만, 실제 미세플라스틱은 환경에서 수 십 년 동안 풍화되므로, 실험조건보다 더욱 노화된 미세플라스틱이 실제 환경에서 유기오염물질과의 상호작용을 통해 더욱 악화된 환경위

In [20]:
index.__dict__

{'_use_async': False,
 '_store_nodes_override': False,
 '_embed_model': OllamaEmbedding(model_name='gemma-2-embed', embed_batch_size=10, callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x7efc1bc2ff10>, num_workers=None, base_url='http://localhost:11434', ollama_additional_kwargs={}),
 '_insert_batch_size': 2048,
 '_storage_context': StorageContext(docstore=<llama_index.core.storage.docstore.simple_docstore.SimpleDocumentStore object at 0x7efc192db350>, index_store=<llama_index.core.storage.index_store.simple_index_store.SimpleIndexStore object at 0x7efc192ce690>, vector_stores={'default': FaissVectorStore(stores_text=False, is_embedding_query=True), 'image': SimpleVectorStore(stores_text=False, is_embedding_query=True, data=SimpleVectorStoreData(embedding_dict={}, text_id_to_ref_doc_id={}, metadata_dict={}))}, graph_store=<llama_index.core.graph_stores.simple.SimpleGraphStore object at 0x7efc19231010>, property_graph_store=None),
 '_docstore': <llama_index.c

In [51]:
type(query_engine)

llama_index.core.query_engine.retriever_query_engine.RetrieverQueryEngine

In [49]:
response_synthesizer = get_response_synthesizer(llm)
query_engine = index.as_query_engine(
   response_mode="compact",
   verbose=True,
   similarity_top_k=1,
   llm=llm,
   response_synthesizer=response_synthesizer
)


In [22]:
type(query_engine)

llama_index.core.query_engine.retriever_query_engine.RetrieverQueryEngine

## Test Query Engine

In [23]:
query_engine.callback_manager

In [34]:

# question = "미세플라스틱이 햇빛에 노출되면 오염줄질을 흡수하는 연구와 관련된 미국 대학교를 찾고있어"
question = "어떤 대학교가 미세플라스틱이 햇빛에 노출되면 오염줄질을 흡수하는 연구하는지 찾아줘"

response = query_engine.query(question + "\n다음 키워드와 관련된 대답을 해줘\n{keyword_response.text} 한글로만 답변해줘")
# response = query_engine.query("태평양의 플라스틱 등 해양쓰레기 청소가 국가 간 무역장벽의 방해를 받고 있다")
display(Markdown(f"<b>{response}</b>"))
display(Markdown(f"\n{'- '}".join([node.text for node in response.source_nodes])))

#### ollama_request_body: {'prompt': '어떤 대학교가 미세플라스틱이 햇빛에 노출되면 오염줄질을 흡수하는 연구하는지 찾아줘\n다음 키워드와 관련된 대답을 해줘\n{keyword_response.text} 한글로만 답변해줘', 'model': 'gemma-2-embed', 'options': {}}


<b>죄송하지만, 미세플라스틱이 노후화되면 오염물질을 더 많이 흡수하는지에 대한 정보는 제공되지 않았습니다. 또한 어떤 대학교가 이러한 연구를 진행하는지도 알 수 없습니다.</b>

연구팀은Chemospere저널에 발표한 논문에서, ‘환경 내에서 오염물질의 매개체로써의 미세플라스틱의 잠재적인 역할을 고려할 때, 미세플라스틱이 야기하는 미래의 환경 위험성을 평가하기 위해서는 표면 풍화와 오염물질 흡착에 대한 복잡한 상호작용을 이해하는 것이 매우 중요하다.‘‘고 결론지었다.
- - 발암성·돌연변이성·생식독성(CMR), 내분비장애(EDC), 잔류성·축척성·독성(PBT) 물질을 포함하여 인체 및 환경에 유해한 화학물질

- 쉽게 재활용할 수 없는 고분자 및 브롬계난연제를 포함하여 안전하고 품질 높은 2차 재료의 재활용성 또는 순환성을 저해하는 물질

- 미세플라스틱과 같이 환경에서 잘 분해되지 않아 방출했을 때 위험성이 있는 물질

> [BUG] Reloading LLM on each query ...
https://stackoverflow.com/questions/78526926/how-to-prevent-llamaindex-queryengine-from-reloading-llm-on-each-query

In [25]:
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.response_synthesizers import BaseSynthesizer

In [26]:
class RAGQueryEngine(CustomQueryEngine):
    """RAG Query Engine."""

    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer

    def custom_query(self, query_str: str):
        nodes = self.retriever.retrieve(query_str)
        response_obj = self.response_synthesizer.synthesize(query_str, nodes)
        return response_obj

In [27]:
retriever = index.as_retriever()

synthesizer = get_response_synthesizer(response_mode="compact")
query_engine = RAGQueryEngine(
    retriever=retriever, response_synthesizer=synthesizer
)

In [28]:
response = query_engine.query("What did the author do growing up?")
print(response.source_nodes[0].get_content())

#### ollama_request_body: {'prompt': 'What did the author do growing up?', 'model': 'gemma-2-embed', 'options': {}}
- 오존층 파괴 및 지구온난화(global warming potential, GWP) 물질

- 고우려 고분자 물질

- 플라스틱 제품으로부터 이동 및 방출 가능성 있는 물질

NGO 단체는 우려되는 화학물질, 고분자 및 플라스틱 제품에 대한 규제 방안 합의에 있어 세부사항의 중요성을 당부함.


# FunctionTool

In [29]:
from llama_index.core.agent import ReActAgent
from llama_index.llms.openai import OpenAI
from llama_index.core.llms import ChatMessage
from llama_index.core.tools import BaseTool, FunctionTool

In [30]:
def multiply(a: int, b: int) -> int:
    """Multiply two integers and returns the result integer"""
    return a * b


multiply_tool = FunctionTool.from_defaults(fn=multiply)

def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b


add_tool = FunctionTool.from_defaults(fn=add)

In [32]:
agent = ReActAgent.from_tools([multiply_tool, add_tool], llm=llm, verbose=True)
response = agent.chat("What is 20+(2*4)? Calculate step by step ")

> Running step 28b95a94-b89d-437d-99e8-c0aa236edde0. Step input: What is 20+(2*4)? Calculate step by step 
Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: add
Action Input: {}
Observation: Error: add() missing 2 required positional arguments: 'a' and 'b'
> Running step 92872d8a-c759-4087-bc06-3637cf66eb15. Step input: None
Thought: (Implicit) I can answer without any more tools!
Answer: I see you made a mistake in your input for the tool! The `add` function requires two arguments, which are represented by the keys "a" and "b" in the Action Input.

You can fix this error like so:
```json
{"a": 20, "b": (2*4)}
```


In [33]:
response.__dict__

{'response': 'I see you made a mistake in your input for the tool! The `add` function requires two arguments, which are represented by the keys "a" and "b" in the Action Input.\n\nYou can fix this error like so:\n```json\n{"a": 20, "b": (2*4)}\n```',
 'sources': [ToolOutput(content="Error: add() missing 2 required positional arguments: 'a' and 'b'", tool_name='add', raw_input={'kwargs': {}}, raw_output=TypeError("add() missing 2 required positional arguments: 'a' and 'b'"), is_error=True)],
 'source_nodes': [],
 'is_dummy_stream': False,
 'metadata': None}